# UdaPlay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside the `games` folder. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [ ]:
# Optional SQLite compatibility for the Udacity workspace.
# pysqlite3 is imported dynamically so local editors do not warn when it is absent.

from importlib import import_module, util
import sys

if util.find_spec("pysqlite3") is not None:
    pysqlite3 = import_module("pysqlite3")
    sys.modules["sqlite3"] = pysqlite3

In [14]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

Create a local `.env` file using `.env.example` as a guide. The real `.env` file is intentionally ignored by Git.

In [15]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")

assert OPENAI_API_KEY is not None, "OPENAI_API_KEY must be set in your local .env file"
assert TAVILY_API_KEY is not None, "TAVILY_API_KEY must be set in your local .env file"

### VectorDB Instance

In [16]:
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [17]:
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    api_base=OPENAI_BASE_URL,
)

In [18]:
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)

### Add documents

In [19]:
# Make sure you have a directory "games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = (
        f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - "
        f"{game['Genre']}, published by {game['Publisher']}. {game['Description']}"
    )

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    # upsert overwrites existing IDs, so re-running re-indexes with the latest content
    collection.upsert(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )
